In [1]:
# Parameters
run_id = "1cc41367-e95c-4686-b4a7-98f690865869"
artifacts_dir = "/home/adnoman/projects/aml_gan/AMLend2end/artifacts/runs/1cc41367-e95c-4686-b4a7-98f690865869"
sample_size = None
epochs = None
threshold = None


## Monitor transactions using anomaly detection model.     
---
**NOTE**: 

In real life scenarios financial transaction are dynamically evolving graphs. Performing anomaly detection inference on graph embeddings in live Transaction Monitoring Systems will require to update the graph and node representations after new transactions arrive. Recomputing entire graph for every newly arrived transaction will lead to unaxeptable delayes and even monitoring system failure. This problem  will be more sever if large amount of updates happen in a short time window.

Contact us at Logical Clocks and we will help you to setup end to end graph based deep anomaly detection for live Transaction Monitoring Systems. 

---

In [2]:
# Setup for local execution
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

# Define paths
BASE_PATH = os.path.dirname(os.path.abspath("__file__"))
TRAINING_DATA_PATH = os.path.join(BASE_PATH, "training_data")
OUTPUT_PATH = os.path.join(BASE_PATH, "output")
MODELS_PATH = os.path.join(BASE_PATH, "models")

print(f"TensorFlow version: {tf.__version__}")

2026-02-02 17:26:10.477689: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-02 17:26:10.506308: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2026-02-02 17:26:11.268575: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow version: 2.20.0


In [3]:
# Find and load the trained anomaly detection model
model_dirs = [d for d in os.listdir(MODELS_PATH) if d.startswith('gan_anomaly_')]
if model_dirs:
    latest_model_dir = os.path.join(MODELS_PATH, sorted(model_dirs)[-1])
    print(f"Found model: {latest_model_dir}")
else:
    raise FileNotFoundError("No trained model found! Run notebook 8 first.")

Found model: /home/adnoman/projects/aml_gan/AMLend2end/models/gan_anomaly_f6ee3607


![Image7-Monitor.png](./images/model_registry.gif)

# Query Model Repository for best anomaly detection model

In [4]:
# Load the model
model_path = os.path.join(latest_model_dir, "anomaly_detector.keras")
model = keras.models.load_model(model_path)
print(f"Loaded model from: {model_path}")

# Load metadata
with open(os.path.join(latest_model_dir, "metadata.json"), 'r') as f:
    metadata = json.load(f)
    
# Load threshold
threshold = np.load(os.path.join(latest_model_dir, "threshold.npy"))
print(f"Anomaly threshold: {threshold:.6f}")

I0000 00:00:1770035171.956191  152054 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9511 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4080 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Loaded model from: /home/adnoman/projects/aml_gan/AMLend2end/models/gan_anomaly_f6ee3607/anomaly_detector.keras
Anomaly threshold: 0.000157


In [5]:
print(f"Model metrics: {metadata['metrics']}")
print(f"Hyperparameters: {metadata['hyperparameters']}")

Model metrics: {'auc': 0.4967773452150562, 'optimal_threshold': 0.00015697844496052933, 'final_loss': 0.0002647935471031815}
Hyperparameters: {'latent_dim': 8, 'n_layers': 2, 'activation': 'relu', 'dropout_rate': 0.0, 'learning_rate': 0.0001}


In [6]:
# Dummy cell

In [7]:
# Dummy cell

### Create the deployment
Here, we fetch the model we want from the model registry and define a configuration for the deployment. For the configuration, we need to specify the serving type (default or KFserving) and in this case, since we use default serving and an sklearn model, we need to give the location of the prediction script.

In [8]:
# Dummy cell

In [9]:
# Dummy cell

In [10]:
# Dummy cell

## Fetch model server object 

In [11]:
# Dummy cell

### Check Model Serving for active deployments
![Image7-Monitor.png](./images/deployment.gif)

In [12]:
# Dummy cell

In [13]:
# Dummy cell

# Retrieve serving vectors and send prediction requests to the served model using Hopsworks REST API

In [14]:
# Load node embeddings for inference
node_embeddings = pd.read_parquet(os.path.join(OUTPUT_PATH, "node_embeddings_fg.parquet"))
emb_cols = [c for c in node_embeddings.columns if c.startswith('emb_')]

print(f"Loaded {len(node_embeddings)} node embeddings")
print(f"Embedding dimensions: {len(emb_cols)}")

Loaded 7347 node embeddings
Embedding dimensions: 32


In [15]:
# Create inference function (replaces model server deployment)
def predict_anomaly(node_id, embeddings_df, model, threshold):
    """Predict if a node is anomalous (potential SAR)."""
    node_data = embeddings_df[embeddings_df['id'] == node_id]
    
    if len(node_data) == 0:
        return {'error': f'Node {node_id} not found'}
    
    # Get embedding
    embedding = node_data[emb_cols].values
    
    # Compute reconstruction error
    reconstructed = model.predict(embedding, verbose=0)
    mse = np.mean(np.square(embedding - reconstructed))
    
    # Determine if anomaly
    is_anomaly = mse > threshold
    
    return {
        'node_id': node_id,
        'anomaly_score': float(mse),
        'threshold': float(threshold),
        'is_anomaly': bool(is_anomaly),
        'prediction': 'POTENTIAL SAR' if is_anomaly else 'NORMAL'
    }

In [16]:
# Sample node IDs for testing
sample_ids = node_embeddings['id'].head(13).tolist()
print(f"Sample node IDs: {sample_ids}")

Sample node IDs: ['3aa9646b', '1e46e726', '49203bc3', 'a74d1101', '616d4505', '99af2455', '39be1ea2', 'e7ec7bdb', 'e2e0d938', 'afc399a9', '75c9a805', 'd7a317f6', 'c14f4989']


In [17]:
# Run inference on sample nodes
print("Running anomaly detection on sample nodes...")
print("=" * 60)

for node_id in sample_ids:
    result = predict_anomaly(node_id, node_embeddings, model, threshold)
    status = "🚨 ALERT" if result['is_anomaly'] else "✓ OK"
    print(f"{status} | Node: {result['node_id']} | Score: {result['anomaly_score']:.6f} | {result['prediction']}")

print("=" * 60)

Running anomaly detection on sample nodes...


2026-02-02 17:26:12.871459: I external/local_xla/xla/service/service.cc:163] XLA service 0x73a768002b00 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-02 17:26:12.871487: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4080 Laptop GPU, Compute Capability 8.9
2026-02-02 17:26:12.875699: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-02-02 17:26:12.895112: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91801
I0000 00:00:1770035173.031750  152228 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


🚨 ALERT | Node: 3aa9646b | Score: 0.000180 | POTENTIAL SAR
🚨 ALERT | Node: 1e46e726 | Score: 0.000292 | POTENTIAL SAR
🚨 ALERT | Node: 49203bc3 | Score: 0.000201 | POTENTIAL SAR
🚨 ALERT | Node: a74d1101 | Score: 0.000278 | POTENTIAL SAR
🚨 ALERT | Node: 616d4505 | Score: 0.000184 | POTENTIAL SAR
🚨 ALERT | Node: 99af2455 | Score: 0.000293 | POTENTIAL SAR


🚨 ALERT | Node: 39be1ea2 | Score: 0.000225 | POTENTIAL SAR
🚨 ALERT | Node: e7ec7bdb | Score: 0.000271 | POTENTIAL SAR
🚨 ALERT | Node: e2e0d938 | Score: 0.000344 | POTENTIAL SAR
🚨 ALERT | Node: afc399a9 | Score: 0.000260 | POTENTIAL SAR
🚨 ALERT | Node: 75c9a805 | Score: 0.000215 | POTENTIAL SAR
✓ OK | Node: d7a317f6 | Score: 0.000151 | NORMAL


🚨 ALERT | Node: c14f4989 | Score: 0.000206 | POTENTIAL SAR


In [18]:
# Run on all nodes and summarize
print("\nRunning on all nodes...")
all_embeddings = node_embeddings[emb_cols].values
reconstructed = model.predict(all_embeddings, verbose=0)
all_scores = np.mean(np.square(all_embeddings - reconstructed), axis=1)

# Add scores to dataframe
node_embeddings['anomaly_score'] = all_scores
node_embeddings['is_anomaly'] = all_scores > threshold

# Summary
print(f"\nTotal nodes: {len(node_embeddings)}")
print(f"Anomalies detected: {node_embeddings['is_anomaly'].sum()}")
print(f"Normal nodes: {(~node_embeddings['is_anomaly']).sum()}")


Running on all nodes...



Total nodes: 7347
Anomalies detected: 7237
Normal nodes: 110


In [19]:
# Show top anomalies
print("\nTop 10 Most Anomalous Nodes:")
top_anomalies = node_embeddings.nlargest(10, 'anomaly_score')[['id', 'is_sar', 'anomaly_score', 'is_anomaly']]
print(top_anomalies.to_string(index=False))

print("\n" + "=" * 50)
print("Anomaly Detection Complete!")
print("=" * 50)


Top 10 Most Anomalous Nodes:
      id  is_sar  anomaly_score  is_anomaly
10b33182       0       0.000458        True
bd776e58       0       0.000457        True
23d87fef       0       0.000453        True
b91da37d       0       0.000450        True
a6a98e30       0       0.000445        True
9a90223f       0       0.000442        True
41621e69       0       0.000440        True
2d1772de       0       0.000435        True
51b1cd26       0       0.000435        True
90da97aa       0       0.000432        True

Anomaly Detection Complete!
